In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS stocks.silver")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
INDICATORS = ["ecb_deposit_rate", "euribor_3m", "euribor_6m", "bond_10y"]

w_seq = Window.partitionBy("indicator").orderBy("date")

# Add daily basis-point change per indicator (rates are in %, so *100 = bps)
df_long = (
    spark.table("stocks.bronze.macro_rates")
    .withColumn("prev_value",        F.lag("value", 1).over(w_seq))
    .withColumn("daily_change_bps",  (F.col("value") - F.col("prev_value")) * 100)
    .drop("prev_value", "indicator_label", "ingested_at")
)

# Pivot to wide format: one row per date, columns per indicator
pivot_value  = df_long.groupBy("date").pivot("indicator", INDICATORS).agg(F.first("value"))
pivot_change = df_long.groupBy("date").pivot("indicator", INDICATORS).agg(F.first("daily_change_bps"))

for ind in INDICATORS:
    pivot_change = pivot_change.withColumnRenamed(ind, f"{ind}_bps_change")

silver_df = pivot_value.join(pivot_change, on="date", how="left").orderBy("date")

In [ ]:
(
    silver_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("stocks.silver.macro_rates")
)

print(f"stocks.silver.macro_rates: {spark.table('stocks.silver.macro_rates').count()} rows")
spark.table("stocks.silver.macro_rates").show(10)